1.Prepare/Pre-process a text corpus to make it more usable for NLP tasks usmg
tokenization, conversion to lowercase, removal of punctuation, filtration of stop
words, stemming and lemmatization.

In [1]:
import nltk 
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

In [2]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /Users/satyamraj/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/satyamraj/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/satyamraj/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/satyamraj/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [4]:
text = "MR.SATYAM RAJPUT from Rajput Clan"

# Tokenization
tokens = word_tokenize(text)

# Lowercase
tokens = [word.lower() for word in tokens]

# Remove punctuation
tokens = [word for word in tokens if word not in string.punctuation]

# Remove stopwords
stop_words = set(stopwords.words('english'))
filtered_tokens = [word for word in tokens if word not in stop_words]

# Stemming
stemmer = PorterStemmer()
stemmed = [stemmer.stem(word) for word in filtered_tokens]

# Lemmatization
lemmatizer = WordNetLemmatizer()
lemmatized = [lemmatizer.lemmatize(word) for word in filtered_tokens]

print("Filtered Tokens:", filtered_tokens)
print("Stemmed:", stemmed)
print("Lemmatized:", lemmatized)

# This except block is likely left over from a previous copy-paste operation and should be removed
# as the try block context is missing in this cell. 
# If needed, it should wrap the relevant code with a 'try' block.
# except LookupError as e:
#     print(f"Resource not found: {e}")
#     print("Please ensure NLTK resources (punkt, stopwords, wordnet) are downloaded. You can run the following commands:")
#     print("nltk.download('punkt')")
#     print("nltk.download('stopwords')")
#     print("nltk.download('wordnet')")

Filtered Tokens: ['mr.satyam', 'rajput', 'rajput', 'clan']
Stemmed: ['mr.satyam', 'rajput', 'rajput', 'clan']
Lemmatized: ['mr.satyam', 'rajput', 'rajput', 'clan']


2.Use regex patterns to extract the usernames from the email addresses, hashtags, dates, and phone numbers present in a given text.

In [5]:
import re

text = """hello mr.satyam rajput
Contact rajputsatyam2921@gmail.com or modi_meloni_23@college.edu.
Call +91 7481878536.
Event on 30/04/2024 and 2024-04-30.
Trending tags: #AI #DataScience #ML
"""

emails = re.findall(r'\b([a-zA-Z0-9._%+-]+)@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b', text)
hashtags = re.findall(r'#\w+', text)
dates = re.findall(r'\b(\d{2}[/-]\d{2}[/-]\d{4}|\d{4}-\d{2}-\d{2})\b', text)
phones = re.findall(r'\b(\+?\d{1,3}[- ]?\d{10})\b', text)

print("Usernames:", emails)
print("Hashtags:", hashtags)
print("Dates:", dates)
print("Phone Numbers:", phones)

Usernames: ['rajputsatyam2921', 'modi_meloni_23']
Hashtags: ['#AI', '#DataScience', '#ML']
Dates: ['30/04/2024', '2024-04-30']
Phone Numbers: ['91 7481878536']


3.List the most common words (with their frequency) in a given text excluding
stopwords.

In [6]:
from collections import Counter
import re

# Example stopwords list (you can expand this)
stopwords = {
    'the', 'is', 'in', 'and', 'to', 'of', 'a', 'an', 'it', 'on', 'for', 'with'
}

def most_common_words(text):
    # Normalize text
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)  # remove punctuation

    # Tokenize
    words = text.split()

    # Remove stopwords
    filtered_words = [word for word in words if word not in stopwords]

    # Count frequency
    word_counts = Counter(filtered_words)

    # Return sorted list
    return word_counts.most_common()

# Example usage
text = "This is a sample text. This text is simple and useful for testing."
print(most_common_words(text))

[('this', 2), ('text', 2), ('sample', 1), ('simple', 1), ('useful', 1), ('testing', 1)]


4.Create the TF-IDF (Term Frequency -Inverse Document Frequency) Matrix for the
given set of text documents

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

docs = [
    "I love IPL",
    "I love CHENNAI SUPER KINGS",
    "CHENNAI SUPER KINGS is FAMILY"
]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(docs)

print(vectorizer.get_feature_names_out())
print(X.toarray())    


['chennai' 'family' 'ipl' 'is' 'kings' 'love' 'super']
[[0.         0.         0.79596054 0.         0.         0.60534851
  0.        ]
 [0.5        0.         0.         0.         0.5        0.5
  0.5       ]
 [0.3935112  0.51741994 0.         0.51741994 0.3935112  0.
  0.3935112 ]]


5. Build a simple statistical language model: estimate unigram and bigram probabilities
with add‐one smoothing and compute the probability of given sentences.

In [8]:
from collections import Counter

def train_unigram(corpus):
    tokens = [w for sent in corpus for w in sent]
    counts = Counter(tokens)
    V = len(counts)
    N = sum(counts.values())
    return counts, V, N

def train_bigram(corpus):
    bigrams = []
    unigram_counts = Counter()
    
    for sent in corpus:
        for i in range(len(sent)-1):
            bigrams.append((sent[i], sent[i+1]))
            unigram_counts[sent[i]] += 1
        unigram_counts[sent[-1]] += 1
    
    return Counter(bigrams), unigram_counts

def bigram_prob(w1, w2, bigram_counts, unigram_counts, V):
    return (bigram_counts[(w1, w2)] + 1) / (unigram_counts[w1] + V)

def sentence_prob(sentence, bigram_counts, unigram_counts, V):
    prob = 1.0
    for i in range(len(sentence)-1):
        prob *= bigram_prob(sentence[i], sentence[i+1],
                            bigram_counts, unigram_counts, V)
    return prob

# corpus
corpus = [
    ["<s>", "I", "love", "NLP", "</s>"],
    ["<s>", "I", "love", "machine", "learning", "</s>"]
]

uni_counts, V, N = train_unigram(corpus)
bi_counts, uni_counts2 = train_bigram(corpus)

sentence = ["<s>", "I", "love", "NLP", "</s>"]
print(sentence_prob(sentence, bi_counts, uni_counts2, V))

0.006172839506172839


6. Perform POS tagging in a given text file. Extract all the nouns present in the text.
Create and print a dictionary with frequency of parts of speech present in the
document.

In [9]:
import nltk
from collections import Counter

nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

def process_text_file(file_path):
    # Read file
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()

    # Tokenize words
    words = nltk.word_tokenize(text)

    # POS tagging
    pos_tags = nltk.pos_tag(words)

    # Extract nouns (NN, NNS, NNP, NNPS)
    nouns = [word for word, tag in pos_tags if tag.startswith('NN')]

    # Count POS frequency
    pos_freq = Counter(tag for _, tag in pos_tags)

    return nouns, dict(pos_freq)

# Example usage
file_path = "note.txt"
nouns, pos_frequency = process_text_file(file_path)

print("Nouns in text:\n", nouns)
print("\nPOS Tag Frequencies:\n", pos_frequency)

Nouns in text:
 ['Introduction', 'Machine', 'Learning', 'Machine', 'Learning', 'ML', 'subset', 'intelligence', 'AI', 'systems', 'experience', 'development', 'algorithms', 'patterns', 'decisions', 'data', 'years', 'ML', 'industries', 'healthcare', 'finance', 'technology', 'Key', 'Concepts', 'Machine', 'Learning', 'Algorithms', 'core', 'machine', 'learning', 'process', 'data', 'learn', 'Common', 'include', 'Learning', 'Algorithms', 'data', 'Examples', 'regression', 'support', 'vector', 'machines', 'SVM', 'networks', 'Learning', 'Algorithms', 'data', 'patterns', 'Examples', 'k-means', 'component', 'analysis', 'PCA', 'Learning', 'Algorithms', 'learn', 'environment', 'receiving', 'rewards', 'penalties', 'Examples', 'Q-learning', 'reinforcement', 'learning', 'Data', 'quality', 'quantity', 'data', 'success', 'ML', 'models', 'Data', 'e.g.', 'images', 'Training', 'Testing', 'Training', 'process', 'data', 'ML', 'algorithm', 'patterns', 'model', 'performance', 'data', 'generalization', 'capabilit

[nltk_data] Downloading package punkt to /Users/satyamraj/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/satyamraj/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


7.Identify and print the named entities using Name Entity Recognition (NER) for a
collection of news headlines.

In [ ]:
# Use ! prefix to run shell commands in Jupyter Notebook
!pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl

In [10]:
import spacy
pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl
   

# Load English model (run: python -m spacy download en_core_web_sm)
nlp = spacy.load("en_core_web_sm")

def extract_entities(headlines):
    results = []

    for headline in headlines:
        doc = nlp(headline)
        entities = [(ent.text, ent.label_) for ent in doc.ents]
        results.append((headline, entities))

    return results

# Example headlines
headlines = [
    "Prime Minister Narendra Modi visits United States for summit",
    "Apple launches new iPhone in California",
    "Microsoft acquires AI startup in London",
    "Floods hit several villages in Assam",
    "Elon Musk announces new Tesla factory in Germany"
]

# Run NER
output = extract_entities(headlines)

# Print results
for headline, entities in output:
    print(f"\nHeadline: {headline}")
    print("Entities:")
    for text, label in entities:
        print(f"  {text} → {label}")

SyntaxError: invalid syntax (2601997340.py, line 2)

8.Classify movie reviews as positive or negative from the IMDB movie dataset of 5OK
movie reviews. (Link for dataset:
https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-review)

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import re

data = pd.read_csv("IMDB Dataset.csv")
X = data["review"]
y = data["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\nTrain: {len(X_train)} | Test: {len(X_test)}")

# TF-IDF vectorization
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=20000
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

# Predictions
y_pred = model.predict(X_test_vec)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))




Train: 40000 | Test: 10000
Accuracy: 0.8947

Classification Report:
              precision    recall  f1-score   support

    negative       0.91      0.88      0.89      4961
    positive       0.88      0.91      0.90      5039

    accuracy                           0.89     10000
   macro avg       0.90      0.89      0.89     10000
weighted avg       0.90      0.89      0.89     10000



9.Build and train a text classifier for the given data (using textbob or simpletransformers
or keras library)

In [ ]:
texts = [
    "I love this product",
    "This is terrible",
    "Amazing experience",
    "Worst service ever"
]

labels = [1, 0, 1, 0]  # 1 = positive, 0 = negative

#Preprocess + Tokenize
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Parameters
vocab_size = 5000
max_len = 50

# Tokenizer
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

# Convert text → sequences
sequences = tokenizer.texts_to_sequences(texts)

# Pad sequences
X = pad_sequences(sequences, maxlen=max_len, padding='post')
y = labels

In [ ]:
#Build on model 

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, GlobalAveragePooling1D

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=16, input_length=max_len),
    GlobalAveragePooling1D(),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')  # binary classification
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()
model.fit(X, y, epochs=10, verbose=1

loss, accuracy = model.evaluate(X, y)
print(f"Accuracy: {accuracy:.2f}")

new_text = ["I really enjoyed this"]
seq = tokenizer.texts_to_sequences(new_text)
padded = pad_sequences(seq, maxlen=max_len, padding='post')

prediction = model.predict(padded)
print("Positive" if prediction[0][0] > 0.5 else "Negative")

10. Generate text using a character-based model using an appropriate dataset. Given a
sequence of characters from a given data ( eg "Shakespear"), train a model to predict
the next character in the sequence ("e").

In [ ]:
with open("shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read().lower()

print("Corpus length:", len(text))
chars = sorted(list(set(text)))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}

vocab_size = len(chars)
print("Unique chars:", vocab_size)
import numpy as np

seq_length = 40
step = 3

sentences = []
next_chars = []

for i in range(0, len(text) - seq_length, step):
    sentences.append(text[i:i+seq_length])
    next_chars.append(text[i+seq_length])

X = np.zeros((len(sentences), seq_length, vocab_size), dtype=np.bool_)
y = np.zeros((len(sentences), vocab_size), dtype=np.bool_)

for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        X[i, t, char_to_idx[char]] = 1
    y[i, char_to_idx[next_chars[i]]] = 1
    
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

model = Sequential([
    LSTM(128, input_shape=(seq_length, vocab_size)),
    Dense(vocab_size, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam')
model.summary()
model.fit(X, y, batch_size=128, epochs=10)
def predict_next_char(model, seed_text):
    seed_text = seed_text.lower()
    
    x_pred = np.zeros((1, seq_length, vocab_size))
    
    for t, char in enumerate(seed_text):
        if char in char_to_idx:
            x_pred[0, t, char_to_idx[char]] = 1
    
    preds = model.predict(x_pred, verbose=0)[0]
    next_index = np.argmax(preds)
    
    return idx_to_char[next_index]

# Example
print(predict_next_char(model, "shakespear"))
def generate_text(model, seed, length=100):
    generated = seed
    
    for _ in range(length):
        next_char = predict_next_char(model, generated[-seq_length:])
        generated += next_char
    
    return generated

print(generate_text(model, "to be or not to be", 100))

Corpus length: 5359444
Unique chars: 70
